# Memory AI Lab — Boundary Detector Training

**Workflow :** éditer le code dans VS Code → `git push` → ouvrir ce notebook dans [colab.research.google.com](https://colab.research.google.com)

**GPU requis** → `Exécution > Modifier le type d'exécution > GPU T4`

## Ce que fait ce notebook

```
Stage 1 — Boundary Detector
   Input  : embeddings mE5-base (768d) + gap temporel
   Modèle : BoundaryMLP 770 → 128 → 32 → 1
   Output : P(frontière) pour chaque message
   Durée  : ~2 min CPU | ~20 sec GPU
```

**Données requises sur Google Drive (`memory_ai_data/`) :**
```
group_anon.txt
group_gold_tune.json
group_gold_test.json
```

Le modèle entraîné est sauvegardé dans `memory_ai_data/boundary_detector.pt`.

In [ ]:
# ── CELLULE 1 : Code depuis GitHub ────────────────────────────────────────
import os, sys
REPO = 'https://github.com/Eloekamaje/memory_ai.git'
CODE_DIR = '/content/memory_ai'
if os.path.exists(CODE_DIR):
    !git -C {CODE_DIR} pull --quiet
else:
    !git clone {REPO} {CODE_DIR} --quiet
sys.path.insert(0, f'{CODE_DIR}/src')
print('✓ Code prêt')

In [ ]:
# ── CELLULE 2 : Dépendances ────────────────────────────────────────────────
!pip install "sympy==1.13.1" -q
!pip install -r {CODE_DIR}/requirements_colab.txt -q
!python -m spacy download fr_core_news_sm -q
print('✓ OK')
print()
print('⚠️  Si première exécution : Exécution > Redémarrer la session,')
print('   puis relancer à partir de la cellule 3.')

In [ ]:
# ── CELLULE 3 : Google Drive + Copie locale (évite les coupures Drive) ───
import os, shutil
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/memory_ai_data'
LOCAL_DIR = '/content/data'
os.makedirs(LOCAL_DIR, exist_ok=True)

FILES_TO_COPY = [
    'group_anon.txt',
    'group_gold_tune.json',
    'group_gold_test.json',
    'group_embeddings_me5.npy',   # optionnel — cache embeddings
]
for fname in FILES_TO_COPY:
    src = f'{DRIVE_DIR}/{fname}'
    dst = f'{LOCAL_DIR}/{fname}'
    if os.path.exists(src) and not os.path.exists(dst):
        print(f'  Copie {fname} ...', end=' ', flush=True)
        shutil.copy2(src, dst)
        print('✓')
    elif os.path.exists(dst):
        print(f'  {fname} déjà en local ✓')
    else:
        print(f'  {fname} absent sur Drive (ignoré)')

DATA_DIR = LOCAL_DIR
print(f'\n✓ DATA_DIR = {DATA_DIR}')

In [ ]:
# ── CELLULE 4 : Parse + Embeddings mE5-base (GPU + cache) ─────────────────
import numpy as np
import torch
from pathlib import Path
from sentence_transformers import SentenceTransformer
from parsers.whatsapp_parser import parse_whatsapp_chat

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {device}')

# mE5-base — multilingual, 768d, handles FR/EN code-switching
MODEL_NAME  = 'intfloat/multilingual-e5-base'
PREFIX      = 'passage: '   # requis par mE5 pour les documents
EMBED_CACHE = Path(DATA_DIR) / 'group_embeddings_me5.npy'

all_artifacts = parse_whatsapp_chat(f'{DATA_DIR}/group_anon.txt')
texts = [PREFIX + a.content for a in all_artifacts]
print(f'[1/2] {len(texts)} messages parsés')

if EMBED_CACHE.exists():
    all_embeddings = np.load(EMBED_CACHE)
    assert len(all_embeddings) == len(texts), 'Cache périmé — supprimer group_embeddings_me5.npy'
    print('[2/2] Embeddings chargés depuis cache')
else:
    print(f'[2/2] Calcul sur {device} (mE5-base 768d) ...')
    model = SentenceTransformer(MODEL_NAME, device=device)
    all_embeddings = model.encode(
        texts, batch_size=256, show_progress_bar=True,
        device=device, convert_to_numpy=True
    ).astype(np.float32)
    np.save(EMBED_CACHE, all_embeddings)
    print(f'      Sauvegardé → {EMBED_CACHE}')

print(f'      Shape : {all_embeddings.shape}  (attendu : (n, 768))')

In [ ]:
# ── CELLULE 5 : Charger tune_early (entraînement boundary detector) ────────
# Protocole B : le boundary detector est entraîné sur la première fraction
# du tune uniquement. La seconde fraction (tune_late) est réservée à Optuna.
# → Aucune fuite de données entre Stage 1 et l'optimisation de Stage 2.
#
# TUNE_SPLIT doit être identique dans 01_eval_ari.ipynb.
TUNE_SPLIT = 0.65   # 65% tune_early pour boundary detector, 35% tune_late pour Optuna

import json

def load_split(path):
    with open(path, encoding='utf-8') as f:
        data = json.load(f)
    n = len(data['artifacts'])
    y_true = [None] * n
    for ep in data['episodes']:
        for idx in range(ep['start_idx'], ep['end_idx'] + 1):
            if idx < n:
                y_true[idx] = ep['episode_id']
    return data['artifacts'], y_true, data['episodes'], data['meta']

tune_arts_all, y_true_tune_all, tune_eps_all, tune_meta = load_split(f'{DATA_DIR}/group_gold_tune.json')
test_arts,     y_true_test,     test_eps,     test_meta  = load_split(f'{DATA_DIR}/group_gold_test.json')

n_tune      = len(tune_arts_all)
n_split     = int(n_tune * TUNE_SPLIT)

arts_all    = all_artifacts[:n_tune]
arts_early  = arts_all[:n_split]
emb_early   = all_embeddings[:n_split]
y_early     = y_true_tune_all[:n_split]

# Test — pour évaluation finale (jamais vu pendant l'entraînement)
arts_test   = all_artifacts[n_tune:n_tune + len(test_arts)]
emb_test    = all_embeddings[n_tune:n_tune + len(test_arts)]

n_early_eps = len(set(y for y in y_early if y is not None))

print(f'Tune total  : {n_tune} msgs · {len(tune_eps_all)} épisodes · {tune_meta["period"]}')
print(f'Tune early  : {n_split} msgs · ~{n_early_eps} épisodes  → boundary detector')
print(f'Tune late   : {n_tune - n_split} msgs                   → Optuna (01_eval_ari.ipynb)')
print(f'Test        : {len(test_arts)} msgs · {len(test_eps)} épisodes · {test_meta["period"]}')

In [ ]:
# ── CELLULE 6 : Labels de frontière (tune_early) ────────────────────────────
# Le TCN travaille directement sur les embeddings — pas de feature extraction.
# On extrait seulement les labels binaires.
from boundary_detector_tcn import extract_boundary_labels
import numpy as np

y_early_bin = extract_boundary_labels(y_early, len(arts_early))
y_test_bin  = extract_boundary_labels(y_true_test, len(arts_test))

n_pos = int(y_early_bin.sum())
n_neg = len(y_early_bin) - n_pos

print(f'Labels tune_early : {n_pos} frontières · {n_neg} continuations')
print(f'Ratio             : 1:{n_neg // max(n_pos,1)} → pos_weight≈{n_neg // max(n_pos,1)}')
print(f'Labels test       : {int(y_test_bin.sum())} frontières / {len(y_test_bin)} msgs')

In [ ]:
# ── CELLULE 7 : Entraînement TCNBoundaryDetector (sur tune_early) ──────────
# Rechargement forcé du module pour prendre en compte les mises à jour git
import importlib, sys
for _mod in list(sys.modules.keys()):
    if 'boundary_detector' in _mod:
        del sys.modules[_mod]

from boundary_detector_tcn import TCNBoundaryDetector

# use_cos_sim=True : ajoute cosine_sim(emb_{t-1}, emb_t) comme feature 770e
# False = baseline 769d (prec≈0.60) | True = +cos_sim 770d (à tester)
USE_COS_SIM = False

detector = TCNBoundaryDetector(
    device=device,
    channels=128,
    n_blocks=3,
    kernel_size=3,
    dropout=0.15,
    window_size=100,
    use_cos_sim=USE_COS_SIM,
)

detector.fit_sequence(
    embeddings  = emb_early,
    artifacts   = arts_early,
    y_true      = y_early,
    n_epochs    = 60,
    lr          = 1e-3,
    weight_decay= 1e-4,
    batch_size  = 32,
    focal_gamma = 2.0,
    stride_min  = 10,
    stride_max  = 25,
    val_split   = 0.10,
    verbose     = True,
)

print(f'\n✓ Entraînement terminé  [use_cos_sim={USE_COS_SIM}]')

In [ ]:
# ── CELLULE 8 : Optimisation seuil sur tune_early (recall ≥ 0.85) ──────────
# Le TCN expose optimize_threshold_sequence() qui itère sur les seuils.
# On cible recall=0.85 (moins conservateur qu'avant) car la précision TCN
# devrait être bien meilleure → on peut se permettre un seuil plus élevé.
MIN_RECALL = 0.85

detector.optimize_threshold_sequence(
    embeddings = emb_early,
    artifacts  = arts_early,
    y_true     = y_early,
    min_recall = MIN_RECALL,
)

print(f'\nSeuil retenu : {detector.threshold:.3f}')

In [ ]:
# ── CELLULE 9 : Évaluation sur TEST (une seule fois) ──────────────────────
import numpy as np

probs_test = detector.predict_proba_sequence(emb_test, arts_test)
preds_test = (probs_test >= detector.threshold).astype(int)

tp   = int(((preds_test == 1) & (y_test_bin == 1)).sum())
fp   = int(((preds_test == 1) & (y_test_bin == 0)).sum())
fn   = int(((preds_test == 0) & (y_test_bin == 1)).sum())
prec = tp / (tp + fp + 1e-8)
rec  = tp / (tp + fn + 1e-8)
f1   = 2 * prec * rec / (prec + rec + 1e-8)
n_b  = int(preds_test.sum())
n_g  = int(y_test_bin.sum())

# Comparaison avec MLP baseline
print(f"""
╔══ TCN BOUNDARY DETECTOR — Résultat TEST ══════════╗
║  Précision  : {prec:.4f}   (MLP baseline ≈ 0.47)  ║
║  Rappel     : {rec:.4f}                            ║
║  F1         : {f1:.4f}                             ║
╠═══════════════════════════════════════════════════╣
║  Frontières : {n_b} prédit / {n_g} gold            ║
║  Seuil      : {detector.threshold:.3f}             ║
╚═══════════════════════════════════════════════════╝

→ Précision >> 0.47 : TCN améliore Stage 1 → gain ARI attendu
→ Précision ≈  0.47 : revoir architecture ou features
""")

In [ ]:
# ── CELLULE 10 : Sauvegarde (local + Drive) ────────────────────────────────
import shutil

LOCAL_PATH = f'{DATA_DIR}/boundary_detector_tcn.pt'
DRIVE_PATH = f'{DRIVE_DIR}/boundary_detector_tcn.pt'

detector.save(LOCAL_PATH)
shutil.copy2(LOCAL_PATH, DRIVE_PATH)

print(f'✓ TCN boundary detector copié vers Drive → {DRIVE_PATH}')
print(f'  Seuil : {detector.threshold:.3f}')
print()
print('Étape suivante : ouvrir 01_eval_ari.ipynb')
print('  USE_HYBRID=True → chargera boundary_detector_tcn.pt en priorité')

---
## Bonus — NSP Coherence Scoring (zéro-shot, BERT multilingue)

Inspiré de CSMSegmenter (SuperDialseg, Jiang et al. 2023).

Principe : BERT NSP prédit P(u_t suit naturellement u_{t-1}).  
Faible cohérence → rupture discursive → frontière probable.

**Zéro-shot** : aucun fine-tuning, uniquement les poids pré-entraînés de `bert-base-multilingual-cased`.

Objectif : mesurer si le signal NSP seul dépasse le TCN (prec≈0.60).  
Si oui → l'ajouter comme feature du TCN (769→770).

In [ ]:
# ── CELLULE 12 : Calcul scores NSP — données complètes (tune + test) ──────
# bert-base-multilingual-cased : 104 langues dont FR et EN
# Durée estimée : ~3 min GPU T4 pour ~11K messages
# Cache sur Drive pour éviter de recalculer
import shutil
from nsp_coherence import compute_nsp_scores, save_nsp_scores, load_nsp_scores

NSP_BACKBONE   = 'bert-base-multilingual-cased'
NSP_CACHE_LOCAL = f'{DATA_DIR}/nsp_scores_mbert.npy'
NSP_CACHE_DRIVE = f'{DRIVE_DIR}/nsp_scores_mbert.npy'

# Textes bruts (sans prefix mE5) pour BERT NSP
all_texts = [a.content for a in all_artifacts]

# Essayer Drive en premier
if not os.path.exists(NSP_CACHE_LOCAL) and os.path.exists(NSP_CACHE_DRIVE):
    shutil.copy2(NSP_CACHE_DRIVE, NSP_CACHE_LOCAL)
    print('NSP scores copiés depuis Drive')

if os.path.exists(NSP_CACHE_LOCAL):
    all_nsp_scores = load_nsp_scores(NSP_CACHE_LOCAL)
    assert len(all_nsp_scores) == len(all_texts)
else:
    all_nsp_scores = compute_nsp_scores(
        all_texts,
        device=device,
        backbone=NSP_BACKBONE,
        batch_size=64,
        max_length=128,
    )
    save_nsp_scores(all_nsp_scores, NSP_CACHE_LOCAL)
    shutil.copy2(NSP_CACHE_LOCAL, NSP_CACHE_DRIVE)
    print(f'✓ Cache sauvegardé sur Drive → {NSP_CACHE_DRIVE}')

# Split test
nsp_test = all_nsp_scores[n_tune:n_tune + len(test_arts)]
print(f'\nNSP scores test : shape={nsp_test.shape} moy={nsp_test[1:].mean():.3f}')

In [ ]:
# ── CELLULE 13 : Évaluation NSP zero-shot sur TEST ────────────────────────
# Balaye cut_rate ∈ [0.1, 2.0] et cherche le meilleur F1
# (équivalent à optimize_threshold mais avec depth-score)
import numpy as np
from nsp_coherence import nsp_boundary_predictions

best_f1, best_cr, best_prec, best_rec = 0.0, 1.0, 0.0, 0.0
best_preds = None

for cut_rate in np.arange(0.1, 3.0, 0.1):
    preds = nsp_boundary_predictions(nsp_test, cut_rate=float(cut_rate))
    tp   = int(((preds == 1) & (y_test_bin == 1)).sum())
    fp   = int(((preds == 1) & (y_test_bin == 0)).sum())
    fn   = int(((preds == 0) & (y_test_bin == 1)).sum())
    p    = tp / (tp + fp + 1e-8)
    r    = tp / (tp + fn + 1e-8)
    f1   = 2 * p * r / (p + r + 1e-8)
    if f1 > best_f1:
        best_f1, best_cr = f1, float(cut_rate)
        best_prec, best_rec = p, r
        best_preds = preds.copy()

n_pred = int(best_preds.sum())
n_gold = int(y_test_bin.sum())

print(f"""
╔══ NSP ZERO-SHOT ({NSP_BACKBONE}) ═══════════════════╗
║  Précision  : {best_prec:.4f}   (TCN baseline ≈ 0.60)      ║
║  Rappel     : {best_rec:.4f}                                ║
║  F1         : {best_f1:.4f}  (cut_rate={best_cr:.1f})       ║
╠══════════════════════════════════════════════════════╣
║  Frontières : {n_pred} prédit / {n_gold} gold               ║
╚══════════════════════════════════════════════════════╝

→ Précision > 0.60 : NSP seul bat le TCN → utiliser comme Stage 1
→ Précision ≈ 0.60 : NSP = TCN → ajouter comme feature (769→770)
→ Précision < 0.60 : NSP seul insuffisant → ajouter comme feature
""")

In [ ]:
# ── CELLULE 14 : Comparaison TCN vs NSP vs combinaison ────────────────────
# Combinaison simple : score final = TCN_prob × (1 - NSP_score)
# (forte prob TCN ET faible cohérence NSP → frontière très probable)
import numpy as np

# TCN seul (déjà calculé en cellule 9)
tcn_probs = detector.predict_proba_sequence(emb_test, arts_test)

# NSP : incoherence = 1 - P(is_next)
nsp_incoherence = 1.0 - nsp_test

# Normalisation min-max pour mettre sur la même échelle
def minmax(x):
    lo, hi = x.min(), x.max()
    return (x - lo) / (hi - lo + 1e-8)

tcn_norm = minmax(tcn_probs)
nsp_norm = minmax(nsp_incoherence)

# Balayage du coefficient de fusion α (0=TCN seul, 1=NSP seul)
print('α     | Prec   | Rec    | F1     | n_pred')
print('-' * 50)
best_combo = {'f1': 0}
for alpha in np.arange(0.0, 1.05, 0.1):
    combined = (1 - alpha) * tcn_norm + alpha * nsp_norm
    # seuil médian (équivalent à ~50% frontières)
    for thr in np.arange(0.2, 0.9, 0.05):
        preds = (combined >= thr).astype(int)
        tp = int(((preds == 1) & (y_test_bin == 1)).sum())
        fp = int(((preds == 1) & (y_test_bin == 0)).sum())
        fn = int(((preds == 0) & (y_test_bin == 1)).sum())
        p  = tp / (tp + fp + 1e-8)
        r  = tp / (tp + fn + 1e-8)
        f1 = 2 * p * r / (p + r + 1e-8)
        if f1 > best_combo['f1']:
            best_combo = {'alpha': float(alpha), 'thr': float(thr),
                          'f1': f1, 'prec': p, 'rec': r,
                          'n_pred': int(preds.sum())}

print(f"Best combo : α={best_combo['alpha']:.1f} | "
      f"prec={best_combo['prec']:.4f} | rec={best_combo['rec']:.4f} | "
      f"F1={best_combo['f1']:.4f} | n_pred={best_combo['n_pred']}")
print()
print('Conclusion :')
print(f'  TCN seul  : prec={prec:.4f} (cellule 9)')
print(f'  NSP seul  : prec={best_prec:.4f} (cellule 13)')
print(f'  Combinaison : prec={best_combo["prec"]:.4f} (α={best_combo["alpha"]:.1f})')
print()
if best_combo['alpha'] > 0.0:
    print('→ NSP apporte de l\'information complémentaire au TCN')
    print('  Prochaine étape : ajouter NSP comme feature (dim 769→770)')
else:
    print('→ NSP n\'améliore pas le TCN sur cette métrique')